In [ ]:
from collections import Counter
import math

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
sent1_list = dataset["sentence1"]
sent2_list = dataset["sentence2"]
labels = dataset["label"]

batch_size = 32
predictions = []
confidences = []
probabilities = []
margins = []
entropies = []
all_logits = []

for start_idx in range(0, len(dataset), batch_size):
    batch_s1 = sent1_list[start_idx:start_idx + batch_size]
    batch_s2 = sent2_list[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch_s1,
        batch_s2,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        sorted_probs, _ = torch.sort(probs, dim=-1, descending=True)
        batch_margins = sorted_probs[:, 0] - sorted_probs[:, 1]
        batch_conf = sorted_probs[:, 0]
        batch_preds = torch.argmax(probs, dim=-1)
        batch_entropy = -(probs * torch.log(probs.clamp(min=1e-12))).sum(dim=-1)
    predictions.extend(batch_preds.cpu().tolist())
    confidences.extend(batch_conf.cpu().tolist())
    probabilities.extend(probs.cpu().tolist())
    margins.extend(batch_margins.cpu().tolist())
    entropies.extend(batch_entropy.cpu().tolist())
    all_logits.extend(logits.cpu().tolist())

print(f"Completed one-pass inference for {len(predictions)} validation examples.")

In [ ]:
full_accuracy = accuracy_score(labels, predictions)
full_precision, full_recall, full_f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
full_cm = confusion_matrix(labels, predictions)

print("Full validation metrics:")
print(f"Accuracy : {full_accuracy:.4f}")
print(f"Precision: {full_precision:.4f}")
print(f"Recall   : {full_recall:.4f}")
print(f"F1       : {full_f1:.4f}")
print("Confusion matrix:")
print(full_cm)
print()
print("Margin/confidence overview on full validation:")
print(f"Mean margin     : {sum(margins) / len(margins):.6f}")
print(f"Min/Max margin  : {min(margins):.6f} / {max(margins):.6f}")
print(f"Mean confidence : {sum(confidences) / len(confidences):.6f}")
print(f"Mean entropy    : {sum(entropies) / len(entropies):.6f}")

In [ ]:
sorted_margins = sorted(margins)
subset_fraction = 0.20
subset_size = max(1, int(len(dataset) * subset_fraction))
margin_threshold = sorted_margins[subset_size - 1]

uncertain_indices = [i for i, m in enumerate(margins) if m <= margin_threshold]

uncertain_rows = []
for i in uncertain_indices:
    uncertain_rows.append({
        "orig_idx": i,
        "sentence1": dataset[i]["sentence1"],
        "sentence2": dataset[i]["sentence2"],
        "label": labels[i],
        "pred_label": predictions[i],
        "correct": int(predictions[i] == labels[i]),
        "confidence": confidences[i],
        "margin": margins[i],
        "entropy": entropies[i],
        "prob_not_paraphrase": probabilities[i][0],
        "prob_paraphrase": probabilities[i][1],
        "logits": all_logits[i]
    })

print(f"Uncertainty subset selection rule: bottom {subset_fraction:.0%} by prediction margin")
print(f"Target subset size (before ties): {subset_size}")
print(f"Margin threshold: {margin_threshold:.6f}")
print(f"Actual subset size (including ties at threshold): {len(uncertain_rows)}")

In [ ]:
if len(uncertain_rows) == 0:
    raise ValueError("Uncertainty subset is empty.")

u_labels = [r["label"] for r in uncertain_rows]
u_preds = [r["pred_label"] for r in uncertain_rows]
u_margins = [r["margin"] for r in uncertain_rows]
u_confidences = [r["confidence"] for r in uncertain_rows]
u_entropies = [r["entropy"] for r in uncertain_rows]
u_correct = [r["correct"] for r in uncertain_rows]

u_accuracy = accuracy_score(u_labels, u_preds)
u_precision, u_recall, u_f1, _ = precision_recall_fscore_support(u_labels, u_preds, average="binary", zero_division=0)
u_cm = confusion_matrix(u_labels, u_preds)

print("Evaluation metrics on uncertainty-defined subset:")
print(f"Accuracy : {u_accuracy:.4f}")
print(f"Precision: {u_precision:.4f}")
print(f"Recall   : {u_recall:.4f}")
print(f"F1       : {u_f1:.4f}")
print("Confusion matrix:")
print(u_cm)
print()
print("Subset margin statistics:")
print(f"Mean margin    : {sum(u_margins) / len(u_margins):.6f}")
print(f"Min/Max margin : {min(u_margins):.6f} / {max(u_margins):.6f}")
print(f"Mean confidence: {sum(u_confidences) / len(u_confidences):.6f}")
print(f"Mean entropy   : {sum(u_entropies) / len(u_entropies):.6f}")
print(f"Correct count  : {sum(u_correct)}")
print(f"Incorrect count: {len(u_correct) - sum(u_correct)}")

In [ ]:
def calibration_summary(rows, n_bins=5):
    bins = []
    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins
        if b < n_bins - 1:
            bucket = [r for r in rows if lo <= r["confidence"] < hi]
        else:
            bucket = [r for r in rows if lo <= r["confidence"] <= hi]
        count = len(bucket)
        if count == 0:
            bins.append({
                "bin": f"[{lo:.1f}, {hi:.1f}{')' if b < n_bins - 1 else ']'}",
                "count": 0,
                "avg_confidence": None,
                "accuracy": None,
                "gap": None
            })
        else:
            avg_conf = sum(r["confidence"] for r in bucket) / count
            acc = sum(r["correct"] for r in bucket) / count
            bins.append({
                "bin": f"[{lo:.1f}, {hi:.1f}{')' if b < n_bins - 1 else ']'}",
                "count": count,
                "avg_confidence": avg_conf,
                "accuracy": acc,
                "gap": abs(avg_conf - acc)
            })
    total = len(rows)
    ece = 0.0
    for item in bins:
        if item["count"] > 0:
            ece += (item["count"] / total) * item["gap"]
    brier = sum((r["prob_paraphrase"] - r["label"]) ** 2 for r in rows) / total
    return bins, ece, brier

full_rows = []
for i in range(len(dataset)):
    full_rows.append({
        "confidence": confidences[i],
        "correct": int(predictions[i] == labels[i]),
        "prob_paraphrase": probabilities[i][1],
        "label": labels[i]
    })

full_bins, full_ece, full_brier = calibration_summary(full_rows, n_bins=5)
u_bins, u_ece, u_brier = calibration_summary(uncertain_rows, n_bins=5)

print("Calibration-style summary on full validation:")
for item in full_bins:
    print(item)
print(f"ECE (5 bins): {full_ece:.6f}")
print(f"Brier score : {full_brier:.6f}")
print()
print("Calibration-style summary on uncertainty subset:")
for item in u_bins:
    print(item)
print(f"ECE (5 bins): {u_ece:.6f}")
print(f"Brier score : {u_brier:.6f}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

lowest_margin_correct = sorted([r for r in uncertain_rows if r["correct"] == 1], key=lambda x: x["margin"])[:10]
lowest_margin_incorrect = sorted([r for r in uncertain_rows if r["correct"] == 0], key=lambda x: x["margin"])[:10]

print(f"Lowest-margin correct examples available: {len(lowest_margin_correct)}")
for i, r in enumerate(lowest_margin_correct, start=1):
    print(f"CORRECT {i}")
    print(f"orig_idx: {r['orig_idx']}")
    print(f"true label: {r['label']} ({label_map[r['label']]})")
    print(f"pred label: {r['pred_label']} ({label_map[r['pred_label']]})")
    print(f"confidence: {r['confidence']:.6f}")
    print(f"margin: {r['margin']:.6f} | entropy: {r['entropy']:.6f}")
    print(f"p(not_paraphrase)={r['prob_not_paraphrase']:.6f} | p(paraphrase)={r['prob_paraphrase']:.6f}")
    print(f"sentence1: {r['sentence1']}")
    print(f"sentence2: {r['sentence2']}")
    print("-" * 100)

print(f"Lowest-margin incorrect examples available: {len(lowest_margin_incorrect)}")
for i, r in enumerate(lowest_margin_incorrect, start=1):
    print(f"INCORRECT {i}")
    print(f"orig_idx: {r['orig_idx']}")
    print(f"true label: {r['label']} ({label_map[r['label']]})")
    print(f"pred label: {r['pred_label']} ({label_map[r['pred_label']]})")
    print(f"confidence: {r['confidence']:.6f}")
    print(f"margin: {r['margin']:.6f} | entropy: {r['entropy']:.6f}")
    print(f"p(not_paraphrase)={r['prob_not_paraphrase']:.6f} | p(paraphrase)={r['prob_paraphrase']:.6f}")
    print(f"sentence1: {r['sentence1']}")
    print(f"sentence2: {r['sentence2']}")
    print("-" * 100)

In [ ]:
full_correct_margins = [margins[i] for i in range(len(dataset)) if predictions[i] == labels[i]]
full_incorrect_margins = [margins[i] for i in range(len(dataset)) if predictions[i] != labels[i]]
u_correct_margins = [r["margin"] for r in uncertain_rows if r["correct"] == 1]
u_incorrect_margins = [r["margin"] for r in uncertain_rows if r["correct"] == 0]

print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset=prediction-uncertainty low-margin subset")
print(f"device={device}")
print(f"num_examples_full={len(dataset)}")
print(f"num_examples_subset={len(uncertain_rows)}")
print(f"subset_fraction_target={subset_fraction:.2f}")
print(f"margin_threshold={margin_threshold:.6f}")
print(f"mean_margin_full={sum(margins) / len(margins):.6f}")
print(f"mean_margin_subset={sum(u_margins) / len(u_margins):.6f}")
print(f"mean_confidence_full={sum(confidences) / len(confidences):.6f}")
print(f"mean_confidence_subset={sum(u_confidences) / len(u_confidences):.6f}")
print(f"mean_entropy_full={sum(entropies) / len(entropies):.6f}")
print(f"mean_entropy_subset={sum(u_entropies) / len(u_entropies):.6f}")
print(f"accuracy_full={full_accuracy:.4f}")
print(f"precision_full={full_precision:.4f}")
print(f"recall_full={full_recall:.4f}")
print(f"f1_full={full_f1:.4f}")
print(f"accuracy_subset={u_accuracy:.4f}")
print(f"precision_subset={u_precision:.4f}")
print(f"recall_subset={u_recall:.4f}")
print(f"f1_subset={u_f1:.4f}")
print(f"ece5_full={full_ece:.6f}")
print(f"ece5_subset={u_ece:.6f}")
print(f"brier_full={full_brier:.6f}")
print(f"brier_subset={u_brier:.6f}")
print(f"mean_margin_correct_full={(sum(full_correct_margins) / len(full_correct_margins)) if full_correct_margins else float('nan'):.6f}")
print(f"mean_margin_incorrect_full={(sum(full_incorrect_margins) / len(full_incorrect_margins)) if full_incorrect_margins else float('nan'):.6f}")
print(f"mean_margin_correct_subset={(sum(u_correct_margins) / len(u_correct_margins)) if u_correct_margins else float('nan'):.6f}")
print(f"mean_margin_incorrect_subset={(sum(u_incorrect_margins) / len(u_incorrect_margins)) if u_incorrect_margins else float('nan'):.6f}")
print(f"num_correct_subset={sum(u_correct)}")
print(f"num_incorrect_subset={len(u_correct) - sum(u_correct)}")